In [62]:
import pandas as pd
from datetime import timedelta
import numpy as np
import matplotlib.pyplot as plt

In [63]:
selected_student = 80

context = pd.read_excel('context/Assessment_Information.xlsx')
context = context[context['student_id'] == selected_student]

context.head()

,student_id,question_id,answer,date,duration,option_selected,Topic,Subtopic,Lecturer_level,Algorithm_level,Typology
17046,80,142,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,1,1,Admin
17047,80,94,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,2,1,Admin
17048,80,734,0,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,3,4,Admin
17049,80,274,1,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,2,1,Admin
17050,80,721,0,2024-01-15T16:37:50.000Z,NaN,NaN,Differentiation,Derivatives,3,4,Admin


In [64]:
context['date'] = pd.to_datetime(context['date'])
last_interaction = context['date'].max()
context['days_since_last_interaction'] = (last_interaction - context['date']).dt.days

step_1 = timedelta(days=15)
step_2 = timedelta(days=30)
step_3 = timedelta(days=60)

# Assign lapse scores based on the days since last interaction: 1 for <= 15 days, 0.6 for 16-30 days, 0.3 for 31-60 days, and 0.1 for > 60 days.
context['lapse_score'] = pd.cut(context['days_since_last_interaction'],
                                   bins=[-1, step_1.days, step_2.days, step_3.days, float('inf')],
                                   labels=['1', '0.6', '0.3', '0.1'])

# sum(difficulty * lapse_score * answer)
context['knowledge_score'] = context['Algorithm_level'] * context['lapse_score'].astype(float) * context['answer']

context['answer'] = context['answer'].replace(-1, 0)

display(context)
print(context['answer'].value_counts())

,student_id,question_id,answer,date,duration,option_selected,Topic,Subtopic,Lecturer_level,Algorithm_level,Typology,days_since_last_interaction,lapse_score,knowledge_score
17046,80,142,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,1,1,Admin,364,0.1,0.1
17047,80,94,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,2,1,Admin,364,0.1,0.1
17048,80,734,0,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,3,4,Admin,364,0.1,0.0
17049,80,274,1,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,2,1,Admin,364,0.1,0.1
17050,80,721,0,2024-01-15 16:37:50+00:00,NaN,NaN,Differentiation,Derivatives,3,4,Admin,364,0.1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19766,80,575,0,2025-01-14 15:58:14+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,0.0
19767,80,1742,0,2025-01-14 15:58:23+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,0.0
19768,80,561,0,2025-01-14 15:58:27+00:00,NaN,NaN,Analytic Geometry,NaN,5,1,Admin,0,1,-1.0
19769,80,1740,1,2025-01-14 15:58:31+00:00,NaN,NaN,Analytic Geometry,NaN,4,1,Admin,0,1,1.0


answer
0    305
1    140
Name: count, dtype: int64


In [65]:
context['lapse_score'] = context['lapse_score'].astype(float)
context['Subtopic'] = context['Subtopic'].fillna(context['Topic'])

new_context = context[['Topic', 'Subtopic', 'lapse_score', 'knowledge_score']]
new_context = new_context.groupby(['Topic', 'Subtopic']).agg({'lapse_score': 'mean', 'knowledge_score': 'sum'}).reset_index()

In [66]:
#new_context.to_csv('context_data.csv', index=False)

In [67]:
new_context

,Topic,Subtopic,lapse_score,knowledge_score
0,Analytic Geometry,Analytic Geometry,0.201449,3.4
1,Complex Numbers,Complex Numbers,0.100000,-0.2
2,Differential Equations,Differential Equations,0.100000,0.9
3,Differentiation,Derivatives,0.162500,2.1
4,Differentiation,Differentiation,0.100000,0.5
5,Differentiation,Implicit Differentiation and Chain Rule,0.100000,0.8
6,Differentiation,Partial Differentiation,0.100000,0.3
7,Discrete Mathematics,Recursivity,0.100000,1.2
8,Discrete Mathematics,Set Theory,0.100000,4.0
9,Fundamental Mathematics,"Algebraic expressions, Equations, and Inequali...",0.100000,1.2


In [68]:
synthetic_data = pd.read_csv('context_data.csv')
display(synthetic_data)

,Topic,Subtopic,lapse_score,knowledge_score
0,Analytic Geometry,Analytic Geometry,0.201449,4.7
1,Complex Numbers,Complex Numbers,0.100000,0.2
2,Differential Equations,Differential Equations,0.100000,1.2
3,Differentiation,Derivatives,0.162500,2.1
4,Differentiation,Differentiation,0.100000,0.5
...,...,...,...,...
1019,Futurology and Tomorrow's Scenarios,End of Privacy or Radical Transparency?,0.100000,0.1
1020,Futurology and Tomorrow's Scenarios,Evolution of the Human Species (Homo Optimus),0.100000,0.1
1021,Futurology and Tomorrow's Scenarios,The Impact of First Contact with Alien Civiliz...,0.100000,0.1
1022,Futurology and Tomorrow's Scenarios,Resource Management on a Earth of 10 Billion P...,0.100000,0.1


In [72]:
min_lapse = synthetic_data['lapse_score'].min()
print(f"Min lapse score: {min_lapse}")
max_lapse = synthetic_data['lapse_score'].max()
print(f"Max lapse score: {max_lapse}")
synthetic_data['lapse_score'] = np.random.normal(loc=synthetic_data['lapse_score'].mean(), scale=2*synthetic_data['lapse_score'].std(), size=len(synthetic_data))
synthetic_data['knowledge_score'] = np.random.normal(loc=synthetic_data['knowledge_score'].mean(), scale=2*synthetic_data['knowledge_score'].std(), size=len(synthetic_data))

display(synthetic_data)

Min lapse score: 0.0244244127626181
Max lapse score: 0.18320289393967187


,Topic,Subtopic,lapse_score,knowledge_score
0,Analytic Geometry,Analytic Geometry,0.081906,0.201424
1,Complex Numbers,Complex Numbers,0.145077,-0.049669
2,Differential Equations,Differential Equations,0.149337,-0.453800
3,Differentiation,Derivatives,0.049352,-0.216439
4,Differentiation,Differentiation,0.084716,-0.005418
...,...,...,...,...
1019,Futurology and Tomorrow's Scenarios,End of Privacy or Radical Transparency?,-0.012934,0.366524
1020,Futurology and Tomorrow's Scenarios,Evolution of the Human Species (Homo Optimus),0.147263,0.515094
1021,Futurology and Tomorrow's Scenarios,The Impact of First Contact with Alien Civiliz...,0.117232,0.814024
1022,Futurology and Tomorrow's Scenarios,Resource Management on a Earth of 10 Billion P...,0.102864,0.444793


In [73]:
synthetic_data['lapse_score'] = pd.qcut(synthetic_data['lapse_score'], q=5, labels={'Extremely lapsed', 'Highly lapsed', 'Moderately lapsed', 'Slightly lapsed', 'Not lapsed'})
synthetic_data['knowledge_score'] = pd.qcut(synthetic_data['knowledge_score'], q=5, labels={'Extremely low knowledge', 'Low knowledge', 'Moderate knowledge', 'High knowledge', 'Extremely high knowledge'})

In [74]:
synthetic_data.to_csv('context_data.csv', index=False)

In [75]:
synthetic_data

,Topic,Subtopic,lapse_score,knowledge_score
0,Analytic Geometry,Analytic Geometry,Slightly lapsed,High knowledge
1,Complex Numbers,Complex Numbers,Moderately lapsed,Extremely low knowledge
2,Differential Equations,Differential Equations,Moderately lapsed,Moderate knowledge
3,Differentiation,Derivatives,Not lapsed,Extremely low knowledge
4,Differentiation,Differentiation,Slightly lapsed,High knowledge
...,...,...,...,...
1019,Futurology and Tomorrow's Scenarios,End of Privacy or Radical Transparency?,Not lapsed,Extremely high knowledge
1020,Futurology and Tomorrow's Scenarios,Evolution of the Human Species (Homo Optimus),Moderately lapsed,Extremely high knowledge
1021,Futurology and Tomorrow's Scenarios,The Impact of First Contact with Alien Civiliz...,Highly lapsed,Low knowledge
1022,Futurology and Tomorrow's Scenarios,Resource Management on a Earth of 10 Billion P...,Extremely lapsed,Extremely high knowledge
